# Module 03 — QC Overview Across Datasets

Cross-dataset quality-control summary for the IVD scRNA-seq atlas (12 datasets).

**Manuscript mapping:** Supplementary Figure S1 — QC metrics across datasets.

This notebook loads preprocessed `.h5ad` files and `qc_summary.tsv` produced by
`scripts/03_preprocessing.py`. No reprocessing is performed here.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='scanpy')

from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
PROC_DIR = BASE / 'data' / 'processed'
QC_DIR = BASE / 'results' / 'qc_reports'
QC_DIR.mkdir(parents=True, exist_ok=True)

ALL_ACCESSIONS = [
    'GSE160756', 'GSE165722', 'GSE189916', 'GSE199866', 'GSE205535',
    'CNP0002664', 'GSE233666', 'GSE244889', 'GSE251686', 'GSE255768',
    'GSE230809', 'GSE242443',
]

# Consistent palette across studies
PALETTE = dict(zip(ALL_ACCESSIONS, sns.color_palette('tab20', len(ALL_ACCESSIONS))))

# ── Load qc_summary.tsv ───────────────────────────────────────────────────
qc_summary_path = QC_DIR / 'qc_summary.tsv'
qc_df = pd.read_csv(qc_summary_path, sep='\t') if qc_summary_path.exists() else pd.DataFrame()
if not qc_df.empty:
    qc_df = qc_df.set_index('accession')
print(f'qc_summary.tsv: {len(qc_df)} datasets')

# ── Load obs DataFrames (no .X) ───────────────────────────────────────────
obs_frames = {}
for acc in ALL_ACCESSIONS:
    path = PROC_DIR / f'{acc}.h5ad'
    if not path.exists():
        print(f'  SKIP {acc}: file not found')
        continue
    adata = sc.read_h5ad(path, backed='r')
    obs_frames[acc] = adata.obs.to_memory() if hasattr(adata.obs, 'to_memory') else adata.obs.copy()
    adata.file.close()
    print(f'  {acc}: {len(obs_frames[acc]):,} cells')

print(f'\nLoaded {len(obs_frames)} datasets')

## Summary Table

In [ ]:
rows = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    cells_after = len(obs)
    cells_raw = int(qc_df.loc[acc, 'cells_raw_total']) if acc in qc_df.index else np.nan
    retention = qc_df.loc[acc, 'retention_pct'] if acc in qc_df.index else np.nan
    rows.append({
        'Accession': acc,
        'Cells (raw)': int(cells_raw) if not np.isnan(cells_raw) else '—',
        'Cells (after QC)': cells_after,
        'Retention (%)': f'{retention:.1f}' if not np.isnan(retention) else '—',
        'Median genes/cell': int(obs['n_genes_by_counts'].median()),
        'Median counts/cell': int(obs['total_counts'].median()),
        'Median %MT': f"{obs['pct_counts_mt'].median():.2f}",
    })

summary_df = pd.DataFrame(rows)

# Totals row
total_raw = sum(r['Cells (raw)'] for r in rows if isinstance(r['Cells (raw)'], int))
total_after = summary_df['Cells (after QC)'].sum()
totals = {
    'Accession': 'TOTAL',
    'Cells (raw)': total_raw,
    'Cells (after QC)': total_after,
    'Retention (%)': '',
    'Median genes/cell': '',
    'Median counts/cell': '',
    'Median %MT': '',
}
summary_df = pd.concat([summary_df, pd.DataFrame([totals])], ignore_index=True)

display(summary_df.style.set_caption('QC Summary — All Datasets').hide(axis='index'))

## Cross-Dataset QC Distributions

In [ ]:
# Build long-form DataFrame for violin plots
qc_long = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    sub = obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].copy()
    sub['dataset'] = acc
    qc_long.append(sub)
qc_long = pd.concat(qc_long, ignore_index=True)

metrics = [
    ('n_genes_by_counts', 'Genes detected', None),
    ('total_counts', 'Total counts (UMI)', None),
    ('pct_counts_mt', '% mitochondrial counts', (0, 15)),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), constrained_layout=True)
for ax, (col, label, ylim) in zip(axes, metrics):
    sns.violinplot(
        data=qc_long, x='dataset', y=col, order=ALL_ACCESSIONS,
        palette=PALETTE, inner='box', linewidth=0.5, cut=0, ax=ax,
        density_norm='width', scale='width' if hasattr(sns, '__version__') else None,
    )
    ax.set_ylabel(label)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    if ylim:
        ax.set_ylim(ylim)
fig.suptitle('Cross-Dataset QC Distributions', fontsize=14, y=1.01)

fig.savefig(QC_DIR / 'notebook_03_qc_violins.png')
plt.show()

## Cells Retained vs Removed

In [ ]:
# Build data for stacked bar chart
bar_data = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    cells_after = len(obs_frames[acc])
    if acc in qc_df.index:
        cells_raw = int(qc_df.loc[acc, 'cells_raw_total'])
        cells_removed = cells_raw - cells_after
    else:
        cells_raw = np.nan
        cells_removed = np.nan
    bar_data.append({'dataset': acc, 'Retained': cells_after, 'Removed': cells_removed})

bar_df = pd.DataFrame(bar_data).set_index('dataset')
# Drop datasets with no raw count info
bar_df_plot = bar_df.dropna()

fig, ax = plt.subplots(figsize=(12, 5))
bar_df_plot[['Retained', 'Removed']].plot.bar(
    stacked=True, ax=ax, color=['#4c72b0', '#dd8452'], edgecolor='white', width=0.7,
)
ax.set_ylabel('Number of cells')
ax.set_xlabel('')
ax.set_title('Cells Retained vs Removed per Dataset')
ax.tick_params(axis='x', rotation=45)
ax.legend(frameon=False)

# Annotate retention %
for i, (idx, row) in enumerate(bar_df_plot.iterrows()):
    total = row['Retained'] + row['Removed']
    pct = row['Retained'] / total * 100
    ax.text(i, total + total * 0.01, f'{pct:.0f}%', ha='center', va='bottom', fontsize=8)

fig.tight_layout()
fig.savefig(QC_DIR / 'notebook_03_cells_retained.png')
plt.show()

## Sequencing Depth Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel A: Box plot of total_counts per dataset
ax = axes[0]
sns.boxplot(
    data=qc_long, x='dataset', y='total_counts', order=ALL_ACCESSIONS,
    palette=PALETTE, fliersize=0.5, linewidth=0.6, ax=ax,
)
ax.set_ylabel('Total counts (UMI per cell)')
ax.set_xlabel('')
ax.set_title('A. Sequencing depth per dataset')
ax.tick_params(axis='x', rotation=45)

# Panel B: Median genes vs median counts per dataset
ax = axes[1]
med_stats = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    med_stats.append({
        'dataset': acc,
        'median_genes': obs['n_genes_by_counts'].median(),
        'median_counts': obs['total_counts'].median(),
    })
med_df = pd.DataFrame(med_stats)
for _, row in med_df.iterrows():
    ax.scatter(row['median_counts'], row['median_genes'],
               color=PALETTE[row['dataset']], s=80, zorder=3)
    ax.annotate(row['dataset'], (row['median_counts'], row['median_genes']),
                fontsize=7, ha='left', va='bottom', xytext=(4, 4),
                textcoords='offset points')
ax.set_xlabel('Median total counts per cell')
ax.set_ylabel('Median genes per cell')
ax.set_title('B. Median genes vs counts per dataset')

fig.tight_layout()
fig.savefig(QC_DIR / 'notebook_03_seq_depth.png')
plt.show()

## Per-Dataset UMAP Panels

In [ ]:
# Load UMAP embeddings for each dataset
umap_data = {}
for acc in ALL_ACCESSIONS:
    path = PROC_DIR / f'{acc}.h5ad'
    if not path.exists():
        continue
    adata = sc.read_h5ad(path, backed='r')
    umap_data[acc] = {
        'umap': adata.obsm['X_umap'][:],  # copy to memory
        'sample_id': obs_frames[acc]['sample_id'].values,
        'leiden': obs_frames[acc]['leiden_res_0.5'].astype(str).values,
        'cell_type': obs_frames[acc]['cell_type_preliminary'].values,
    }
    adata.file.close()

datasets = [acc for acc in ALL_ACCESSIONS if acc in umap_data]
n_datasets = len(datasets)
color_keys = ['sample_id', 'leiden', 'cell_type']
col_titles = ['Sample', 'Leiden cluster', 'Preliminary cell type']

fig, axes = plt.subplots(n_datasets, 3, figsize=(18, 4 * n_datasets))
if n_datasets == 1:
    axes = axes[np.newaxis, :]

for row_i, acc in enumerate(datasets):
    umap = umap_data[acc]['umap']
    for col_i, (key, title) in enumerate(zip(color_keys, col_titles)):
        ax = axes[row_i, col_i]
        labels = umap_data[acc][key]
        unique_labels = sorted(set(labels), key=str)
        n_colors = len(unique_labels)
        cpal = sns.color_palette('tab20', n_colors) if n_colors <= 20 else sns.color_palette('husl', n_colors)
        color_map = dict(zip(unique_labels, cpal))

        # Shuffle points for better overlap rendering
        idx = np.random.RandomState(42).permutation(len(umap))
        colors = [color_map[labels[i]] for i in idx]
        ax.scatter(umap[idx, 0], umap[idx, 1], c=colors, s=0.5, alpha=0.6, rasterized=True)
        ax.set_xticks([])
        ax.set_yticks([])
        if row_i == 0:
            ax.set_title(title, fontsize=11)
        if col_i == 0:
            ax.set_ylabel(acc, fontsize=10, fontweight='bold')

fig.suptitle('Per-Dataset UMAP Panels', fontsize=14, y=1.0)
fig.tight_layout()
fig.savefig(QC_DIR / 'notebook_03_umap_panels.png', dpi=150)
plt.show()

## Notes

- QC filtering applied: `n_genes_by_counts` ≥ 200, `pct_counts_mt` ≤ 20%, doublet removal via Scrublet.
- Datasets missing from `qc_summary.tsv` show "—" for raw cell counts.
- Preliminary cell type labels are based on marker-gene scoring; definitive annotation follows in Module 04.
- Figures saved to `results/qc_reports/` with prefix `notebook_03_`.

In [ ]:
saved_figures = sorted(QC_DIR.glob('notebook_03_*.png'))
print('Saved figures:')
for f in saved_figures:
    print(f'  {f.relative_to(BASE)}')